# AAI-590 Capstone | Assignment 3.1 — Data Summary
## FoundationalASSIST: Data Cleaning & Exploratory Data Analysis

**Author:** Monish Yarapathineni  
**Dataset:** FoundationalASSIST (Worden et al., 2026) — [HuggingFace](https://huggingface.co/datasets/ASSISTments/FoundationalASSIST)  
**License:** CC-BY-NC-4.0  

> This notebook covers (1) loading the three dataset splits, (2) cleaning and type-casting, and (3) exploratory analysis of the behavioral signals, content structure, and student sequence properties relevant to misconception classification.


## 0. Setup & Imports

In [1]:
# Install datasets library if running on Colab
# (pandas, numpy, matplotlib, seaborn are pre-installed on Colab)
try:
    import google.colab
    IN_COLAB = True
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'datasets', 'huggingface_hub'], check=True)
except ImportError:
    IN_COLAB = False

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datasets import load_dataset, load_from_disk

# ── Styling ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (9, 4)})

print(f'Running on Colab: {IN_COLAB}')


ModuleNotFoundError: No module named 'datasets'

## 1. Load Data

The dataset ships as three Arrow-backed HuggingFace `DatasetDict` splits stored locally. We convert each to a pandas DataFrame for analysis.

In [ ]:
# ── Authenticate with HuggingFace (required for gated dataset) ──────
# On Colab: enter your HF token when prompted.
# Locally: run `huggingface-cli login` once in your terminal instead.
if IN_COLAB:
    from huggingface_hub import login
    login()  # prompts for token; dataset is gated (CC-BY-NC-4.0)

# ── Load from HuggingFace (Colab) or local disk ──────────────────────
HF_REPO = 'ASSISTments/FoundationalASSIST'

# Local path — update if your data lives elsewhere
LOCAL_BASE = os.path.expanduser(
    '~/Desktop/Educator/aai590-capstone/data/foundationalassist'
)

if IN_COLAB:
    problems_ds     = load_dataset(HF_REPO, 'Foundational ASSIST Dataset')
    interactions_ds = load_dataset(HF_REPO, 'Interactions')
    skills_ds       = load_dataset(HF_REPO, 'Skills')
else:
    problems_ds     = load_from_disk(os.path.join(LOCAL_BASE, 'Foundational ASSIST Dataset'))
    interactions_ds = load_from_disk(os.path.join(LOCAL_BASE, 'interactions'))
    skills_ds       = load_from_disk(os.path.join(LOCAL_BASE, 'skills'))

problems     = problems_ds['train'].to_pandas()
interactions = interactions_ds['train'].to_pandas()
skills       = skills_ds['train'].to_pandas()

print(f'Problems:     {problems.shape}')
print(f'Interactions: {interactions.shape}')
print(f'Skills:       {skills.shape}')


## 2. Dataset Overview

In [ ]:
print("=== PROBLEMS columns ===")
print(problems.dtypes)
print()
print("=== INTERACTIONS columns ===")
print(interactions.dtypes)
print()
print("=== SKILLS columns ===")
print(skills.dtypes)


In [ ]:
# Missing value summary
def null_report(df, name):
    n = df.isnull().sum()
    pct = (n / len(df) * 100).round(2)
    report = pd.DataFrame({'null_count': n, 'null_pct': pct})
    print(f"\n=== {name} ({len(df):,} rows) ===")
    print(report[report['null_count'] > 0].to_string())

null_report(problems,     "Problems")
null_report(interactions, "Interactions")
null_report(skills,       "Skills")


## 3. Data Cleaning

### 3.1 Parse `end_time`
The `end_time` column uses mixed ISO-8601 formats — some rows include microseconds (`2019-08-25 22:52:54.873+00`), others do not (`2020-04-07 17:12:31+00`). We use `format='mixed'` to handle both.


In [ ]:
interactions['end_time'] = pd.to_datetime(
    interactions['end_time'], format='mixed', utc=True
)
print("Parsed end_time dtype:", interactions['end_time'].dtype)
print("Range:", interactions['end_time'].min().date(), "→", interactions['end_time'].max().date())


### 3.2 Drop the unnamed index column

In [ ]:
# 'Unnamed: 0' is the original DataFrame index re-serialised; drop it
if 'Unnamed: 0' in interactions.columns:
    interactions.drop(columns=['Unnamed: 0'], inplace=True)
    print("Dropped 'Unnamed: 0'")
print("Interactions columns:", list(interactions.columns))


### 3.3 Cast types

In [ ]:
interactions['hint_count']    = interactions['hint_count'].astype('int8')
interactions['saw_answer']    = interactions['saw_answer'].astype('bool')
interactions['discrete_score'] = pd.to_numeric(interactions['discrete_score'], errors='coerce')

# Derive session year for trend analysis
interactions['year'] = interactions['end_time'].dt.year.astype('Int64')  # nullable int

print("Final interactions dtypes:")
print(interactions.dtypes)


### 3.4 Referential integrity check

In [ ]:
i_ids = set(interactions['problem_id'].unique())
p_ids = set(problems['problem_id'].unique())
s_ids = set(skills['problem_id'].unique())

print(f"Problems in interactions:          {len(i_ids):,}")
print(f"Problems in problems table:        {len(p_ids):,}")
print(f"Problems with skill tags:          {len(s_ids):,}")
print(f"Orphaned (in interactions, no meta): {len(i_ids - p_ids):,}")
print(f"Never attempted (in meta, no inter): {len(p_ids - i_ids):,}")


## 4. Student Sequence Analysis

For an LSTM sequence model, sequence length distribution determines truncation/padding decisions.

In [ ]:
seq_len = interactions.groupby('user_id').size().rename('seq_length')

print(f"Unique students: {seq_len.shape[0]:,}")
print()
print(seq_len.describe().round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(seq_len, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel("Interactions per Student")
axes[0].set_ylabel("Count")
axes[0].set_title("Sequence Length Distribution")
axes[0].axvline(seq_len.mean(), color='tomato', linestyle='--', label=f'Mean {seq_len.mean():.0f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(seq_len, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5))
axes[1].set_ylabel("Interactions per Student")
axes[1].set_title("Sequence Length Boxplot")
axes[1].set_xticks([])

plt.suptitle("Student Sequence Lengths", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("sequence_lengths.png", bbox_inches='tight')
plt.show()


In [ ]:
# Activity over time
yearly = interactions.groupby('year').size()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(yearly.index, yearly.values, color='steelblue', edgecolor='white')
ax.set_xlabel("Year")
ax.set_ylabel("Number of Interactions")
ax.set_title("Interaction Volume by Year")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig("interactions_by_year.png", bbox_inches='tight')
plt.show()


## 5. Behavioral Signals

`discrete_score`, `hint_count`, and `saw_answer` are the three behavioral features that will be core inputs to the sequence classifier.

In [ ]:
scored = interactions.dropna(subset=['discrete_score'])
acc = scored['discrete_score'].mean()

print(f"Interactions with score:  {len(scored):,}  ({len(scored)/len(interactions):.1%})")
print(f"Null discrete_score:      {interactions['discrete_score'].isna().sum():,}")
print(f"Overall accuracy:         {acc:.3f}  ({acc:.1%})")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# discrete_score distribution
score_counts = scored['discrete_score'].value_counts().sort_index()
axes[0].bar(score_counts.index.astype(str), score_counts.values, color='steelblue', edgecolor='white')
axes[0].set_xlabel("discrete_score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score Distribution")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))

# hint_count distribution (0 vs 1-4)
hint_counts = interactions['hint_count'].value_counts().sort_index()
axes[1].bar(hint_counts.index.astype(str), hint_counts.values, color='teal', edgecolor='white')
axes[1].set_xlabel("hint_count")
axes[1].set_ylabel("Count")
axes[1].set_title("Hint Count Distribution")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))

# saw_answer rate
sa_counts = interactions['saw_answer'].value_counts()
axes[2].bar(['Not Seen', 'Saw Answer'], [sa_counts.get(False,0), sa_counts.get(True,0)],
            color=['steelblue', 'tomato'], edgecolor='white')
axes[2].set_ylabel("Count")
axes[2].set_title(f"Saw Answer\n({interactions['saw_answer'].mean():.1%} of interactions)")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))

plt.suptitle("Behavioral Signals", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("behavioral_signals.png", bbox_inches='tight')
plt.show()


### 5.1 Correlation between behavioral features

In [ ]:
corr_df = interactions[['hint_count', 'saw_answer', 'discrete_score']].copy()
corr_df['saw_answer'] = corr_df['saw_answer'].astype(float)
corr_df = corr_df.dropna()

corr_matrix = corr_df.corr()
print(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(5.5, 4))
mask = np.zeros_like(corr_matrix)
mask[np.triu_indices_from(mask, k=1)] = True   # upper triangle (keep lower + diag)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title("Pearson Correlation — Behavioral Features", fontweight='bold')
plt.tight_layout()
plt.savefig("correlation_heatmap.png", bbox_inches='tight')
plt.show()


## 6. Problem Content Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Problem type
pt_counts = problems['Problem Type'].value_counts()
axes[0].barh(pt_counts.index[::-1], pt_counts.values[::-1], color='steelblue', edgecolor='white')
axes[0].set_xlabel("Number of Problems")
axes[0].set_title("Problem Type Distribution")
for i, v in enumerate(pt_counts.values[::-1]):
    axes[0].text(v + 10, i, str(v), va='center', fontsize=9)

# Answer type
at_counts = problems['Answer Types'].value_counts().head(7)
axes[1].barh(at_counts.index[::-1], at_counts.values[::-1], color='teal', edgecolor='white')
axes[1].set_xlabel("Number of Problems")
axes[1].set_title("Answer Type Distribution (Top 7)")
for i, v in enumerate(at_counts.values[::-1]):
    axes[1].text(v + 5, i, str(v), va='center', fontsize=9)

plt.suptitle("Problem Content Structure", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("problem_types.png", bbox_inches='tight')
plt.show()


In [ ]:
# Grade level distribution (from node_code prefix)
skills['grade'] = skills['node_code'].str.extract(r'^(\d+)').astype(float)
grade_counts = skills['grade'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar(grade_counts.index.astype(int).astype(str), grade_counts.values,
              color='steelblue', edgecolor='white')
ax.set_xlabel("Grade Level")
ax.set_ylabel("Number of Skills/KCs")
ax.set_title("Knowledge Component Distribution by Grade Level")
for bar, v in zip(bars, grade_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig("grade_distribution.png", bbox_inches='tight')
plt.show()

print("Grade counts:")
for g, c in grade_counts.items():
    print(f"  Grade {int(g)}: {c:,} KCs")


## 7. Knowledge Component (KC) Difficulty

Skill-level accuracy reveals which knowledge components are hardest — these are likely candidates for concentrated misconception signal.

In [ ]:
merged = interactions.merge(skills, on='problem_id', how='left')
kc_stats = (
    merged.groupby('node_name')['discrete_score']
    .agg(['mean', 'count'])
    .query('count > 100')
    .rename(columns={'mean': 'accuracy', 'count': 'n_interactions'})
    .sort_values('accuracy')
)
print(f"KCs with >100 interactions: {len(kc_stats):,}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hardest KCs
hardest = kc_stats.head(10)
labels_h = [n[:45] + '…' if len(n) > 45 else n for n in hardest.index]
bars = axes[0].barh(labels_h[::-1], hardest['accuracy'].values[::-1],
                    color='tomato', edgecolor='white')
axes[0].set_xlabel("Mean Accuracy")
axes[0].set_title("10 Hardest KCs (>100 interactions)")
axes[0].set_xlim(0, 1)
for bar, v in zip(bars, hardest['accuracy'].values[::-1]):
    axes[0].text(v + 0.01, bar.get_y() + bar.get_height()/2, f'{v:.2f}', va='center', fontsize=8)

# Easiest KCs
easiest = kc_stats.tail(10)
labels_e = [n[:45] + '…' if len(n) > 45 else n for n in easiest.index]
bars = axes[1].barh(labels_e[::-1], easiest['accuracy'].values[::-1],
                    color='steelblue', edgecolor='white')
axes[1].set_xlabel("Mean Accuracy")
axes[1].set_title("10 Easiest KCs (>100 interactions)")
axes[1].set_xlim(0, 1)
for bar, v in zip(bars, easiest['accuracy'].values[::-1]):
    axes[1].text(v + 0.01, bar.get_y() + bar.get_height()/2, f'{v:.2f}', va='center', fontsize=8)

plt.suptitle("Knowledge Component Difficulty", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("kc_difficulty.png", bbox_inches='tight')
plt.show()


## 8. Per-Student Accuracy Distribution

Variation in per-student accuracy informs whether misconceptions are widespread or student-specific.

In [ ]:
student_acc = (
    interactions.dropna(subset=['discrete_score'])
    .groupby('user_id')['discrete_score']
    .mean()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(student_acc, bins=50, color='steelblue', edgecolor='white')
ax.axvline(student_acc.mean(), color='tomato', linestyle='--',
           label=f'Mean {student_acc.mean():.2f}')
ax.axvline(student_acc.median(), color='darkorange', linestyle=':',
           label=f'Median {student_acc.median():.2f}')
ax.set_xlabel("Per-Student Mean Accuracy")
ax.set_ylabel("Number of Students")
ax.set_title("Distribution of Per-Student Accuracy")
ax.legend()
plt.tight_layout()
plt.savefig("per_student_accuracy.png", bbox_inches='tight')
plt.show()

print(student_acc.describe().round(3))


## 9. Summary of Key Findings

| Dimension | Finding |
|---|---|
| Dataset size | 1,753,384 interactions across 5,000 students and 3,368 problems |
| Date range | January 2019 – July 2024 |
| Sequence lengths | All students: 215–421 interactions, mean ≈ 351 (very uniform) |
| Overall accuracy | 61.6% (null scores: 0.51% of rows) |
| Hint usage | 5.2% of interactions use ≥1 hint; max 4 hints |
| Saw answer | 21.7% of interactions — strong negative predictor of correctness (r = −0.668) |
| Dominant problem types | Fill-in-the-blank (64%), Multiple Choice (35%) |
| Grade coverage | Primarily grades 6–8 (middle school), with sparse grades 2–5 |
| KC difficulty range | 14.1% (Write & Interpret Expressions) to 87.9% (Divide Whole Numbers) |
| Referential integrity | Perfect — all 3,368 interacted problems have metadata and skill tags |

### Implications for Modeling
- Uniform sequence lengths simplify batching but suggest a curated dataset (not naturalistic)
- The `saw_answer` signal is the strongest behavioral predictor of failure; it will be a key LSTM input feature
- KC difficulty variance is large (14%–88%), making `node_name` a useful curriculum-structure feature
- Hint count is heavily zero-skewed — will need to decide between treating it as numeric vs. ordinal category


## 10. Stage 1 Readiness Analysis

These cells define and characterise the **genuine misconception rows** — the subset of interactions that Stage 1 (LLM labeling) will operate on — and estimate the labeling workload.

**Definition:** a genuine misconception row is one where:
- `discrete_score == 0` (student did not get full credit on first attempt), AND
- `answer_text ≠ correct answer` (the student's first typed answer was actually wrong)

Rows where `answer_text == correct answer` but `discrete_score == 0` are **hint-penalty rows** — the student answered correctly but had already requested a hint or saw the answer. These are NOT misconception rows and are excluded from Stage 1 labeling.


### 10.1 Text Cleaning (HTML + MathML → plain text)

In [ ]:
import re

def strip_html(text):
    """Strip HTML tags, MathML markup, and common HTML entities.
    Used to clean Problem Body and Multiple Choice Options before LLM input.
    """
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove MathML namespace declarations
    text = re.sub(r'xmlns="[^"]*"', '', text)
    # Replace common MathML elements with readable equivalents
    text = re.sub(r'<mfrac>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</mfrac>', r'\1/\2', text)
    text = re.sub(r'<msup>\s*<mi>([^<]+)</mi>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<msup>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<mo>([^<]+)</mo>', r' \1 ', text)   # operators
    text = re.sub(r'<m[a-z]+>([^<]*)</m[a-z]+>', r'\1', text)  # remaining math tags
    # Remove all remaining HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # HTML entities
    replacements = {'&nbsp;': ' ', '&lt;': '<', '&gt;': '>', '&amp;': '&',
                    '&le;': '≤', '&ge;': '≥', '&deg;': '°', '&times;': '×'}
    for ent, char in replacements.items():
        text = text.replace(ent, char)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'&[a-z]+;', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def normalise_answer(s):
    """Lowercase + strip for answer comparison."""
    if pd.isna(s): return ""
    return str(s).strip().lower()

# Test
sample_math = '<p><math xmlns="http://www.w3.org/1998/Math/MathML"><mn>212</mn><mo>-</mo><mfrac><mn>1</mn><mn>500</mn></mfrac><mi>e</mi><mo>&lt;</mo><mn>195</mn></math></p>'
print('Before:', sample_math[:80])
print('After: ', strip_html(sample_math))


### 10.2 Merge Problems onto Interactions

In [ ]:
# Merge problem metadata needed for misconception filtering
merged = interactions.merge(
    problems[['problem_id', 'Problem Type', 'Answer Types',
              'Fill-in Answers', 'Multiple Choice Answers',
              'Multiple Choice Options', 'Problem Body']],
    on='problem_id', how='left'
)

fillin_mask = merged['Problem Type'] == 'Fill-in-the-blank(s)'
mc_mask     = merged['Problem Type'].isin(
    ['Multiple Choice (select 1)', 'Multiple Choice (select all)']
)
print(f'Merged shape: {merged.shape}')


### 10.3 Misconception Row Count

Apply the filter and quantify how many rows are genuine misconception rows vs. hint-penalty rows.


In [ ]:
# Fill-in genuine misconceptions: score=0 AND student answer ≠ correct answer
fillin_wrong = merged[
    fillin_mask &
    (merged['discrete_score'] == 0) &
    merged.apply(
        lambda r: normalise_answer(r['answer_text']) != normalise_answer(r['Fill-in Answers']),
        axis=1
    )
]

# MC genuine misconceptions: score=0 AND student answer ≠ correct option
mc_wrong = merged[
    mc_mask &
    (merged['discrete_score'] == 0) &
    merged.apply(
        lambda r: normalise_answer(r['answer_text']) != normalise_answer(r['Multiple Choice Answers']),
        axis=1
    )
]

# Hint-penalty rows (answered correctly but penalised for hint/saw_answer usage)
fillin_hint_penalty = merged[
    fillin_mask & (merged['discrete_score'] == 0) &
    merged.apply(
        lambda r: normalise_answer(r['answer_text']) == normalise_answer(r['Fill-in Answers']),
        axis=1
    )
]
mc_hint_penalty = merged[
    mc_mask & (merged['discrete_score'] == 0) &
    merged.apply(
        lambda r: normalise_answer(r['answer_text']) == normalise_answer(r['Multiple Choice Answers']),
        axis=1
    )
]

total = len(merged)
total_misc = len(fillin_wrong) + len(mc_wrong)

summary = pd.DataFrame({
    'Category': [
        'Fill-in genuine wrong (misconception)',
        'Fill-in hint-penalty (answered correctly, penalised)',
        'MC genuine wrong (misconception)',
        'MC hint-penalty (answered correctly, penalised)',
    ],
    'Count': [len(fillin_wrong), len(fillin_hint_penalty), len(mc_wrong), len(mc_hint_penalty)],
})
summary['% of all interactions'] = (summary['Count'] / total * 100).round(2)
print(summary.to_string(index=False))
print(f'\nTOTAL genuine misconception rows: {total_misc:,}  ({total_misc/total:.1%} of all interactions)')


### 10.4 MC Distractor Analysis

In [ ]:
mc_probs = problems[problems['Problem Type'] == 'Multiple Choice (select 1)'].copy()
mc_probs['n_options'] = mc_probs['Multiple Choice Options'].str.split(r'\|\|').apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# Image vs text-only options
has_img = mc_probs['Multiple Choice Options'].str.contains('<img', na=False)
n_img   = has_img.sum()
n_text  = (~has_img).sum()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Options per problem
opt_counts = mc_probs['n_options'].value_counts().sort_index()
axes[0].bar(opt_counts.index.astype(str), opt_counts.values, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Number of answer options')
axes[0].set_ylabel('Number of problems')
axes[0].set_title('MC: Answer Options per Problem')
for i, v in enumerate(opt_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=9)

# Image vs text
axes[1].bar(['Text-only options', 'Image options'], [n_text, n_img],
            color=['steelblue', 'tomato'], edgecolor='white')
axes[1].set_ylabel('Number of MC problems')
axes[1].set_title('MC: Labelable by LLM (text) vs. Not (image)')
for i, v in enumerate([n_text, n_img]):
    axes[1].text(i, v + 1, f'{v} ({v/len(mc_probs):.1%})', ha='center', fontsize=9)

plt.suptitle('Multiple Choice Problem Structure', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mc_distractor_analysis.png', bbox_inches='tight')
plt.show()

print(f'MC problems with text-only options (LLM-labelable): {n_text} ({n_text/len(mc_probs):.1%})')
print(f'MC problems with image options (skipped in Stage 1): {n_img} ({n_img/len(mc_probs):.1%})')


### 10.5 Fill-in Wrong Answer Coverage

For each fill-in problem, how many of the wrong attempts are covered by labeling the top N most common wrong answers?
We use **top 10** as our Stage 1 threshold.


In [ ]:
def top_n_coverage(group, n):
    top = group['answer_text'].value_counts().head(n).sum()
    return top / len(group)

coverage = {}
for n in [3, 5, 10, 20]:
    cov = fillin_wrong.groupby('problem_id').apply(
        top_n_coverage, n=n, include_groups=False
    )
    coverage[n] = cov

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Mean coverage bar chart
ns = list(coverage.keys())
means = [coverage[n].mean() for n in ns]
medians = [coverage[n].median() for n in ns]
x = range(len(ns))
axes[0].bar([i - 0.2 for i in x], means,   width=0.35, label='Mean',   color='steelblue', edgecolor='white')
axes[0].bar([i + 0.2 for i in x], medians, width=0.35, label='Median', color='teal',      edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Top {n}' for n in ns])
axes[0].set_ylabel('Coverage of wrong attempts')
axes[0].set_title('Fill-in: Wrong Attempt Coverage by Top-N Labeling')
axes[0].set_ylim(0, 1)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[0].legend()
for i, (m, med) in enumerate(zip(means, medians)):
    axes[0].text(i - 0.2, m + 0.01, f'{m:.0%}', ha='center', fontsize=8)
    axes[0].text(i + 0.2, med + 0.01, f'{med:.0%}', ha='center', fontsize=8)

# Distribution of unique wrong answers per problem
per_prob_unique = fillin_wrong.groupby('problem_id')['answer_text'].nunique()
axes[1].hist(per_prob_unique, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(10, color='tomato', linestyle='--', label='Top-10 threshold')
axes[1].set_xlabel('Unique wrong answers per problem')
axes[1].set_ylabel('Number of problems')
axes[1].set_title('Fill-in: Distribution of Unique Wrong Answers')
axes[1].legend()

plt.suptitle('Fill-in Answer Coverage Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fillin_coverage.png', bbox_inches='tight')
plt.show()

for n in ns:
    print(f'Top {n:2d}: mean {coverage[n].mean():.1%}, median {coverage[n].median():.1%}')
print(f'\nDecision: label top 10 wrong answers per fill-in problem')


### 10.6 Stage 1 Labeling Volume Estimate

In [ ]:
# MC: unique (problem_id, distractor_text) pairs from text-only problems
mc_text_ids = mc_probs[~mc_probs['Multiple Choice Options'].str.contains('<img', na=False)]['problem_id']
mc_wrong_text = mc_wrong[mc_wrong['problem_id'].isin(mc_text_ids)]
mc_unique_pairs = mc_wrong_text.groupby(['problem_id', 'answer_text']).ngroups

# Fill-in: top 10 wrong answers per problem
fillin_top10 = (
    fillin_wrong
    .groupby('problem_id', group_keys=False)
    .apply(lambda g: g.nlargest(10, 'problem_id')[['problem_id','answer_text']]  # placeholder — we just want top 10 unique
           if False else
           g['answer_text'].value_counts().head(10).reset_index().assign(problem_id=g.name))
)
fillin_label_pairs = len(fillin_top10)

print('Stage 1 labeling volume:')
print(f'  MC unique (problem, distractor) pairs : {mc_unique_pairs:>6,}')
print(f'  Fill-in top-10 (problem, answer) pairs: {fillin_label_pairs:>6,}')
print(f'  Total pairs to label                  : {mc_unique_pairs + fillin_label_pairs:>6,}')
print()
batch_size = 50
total_pairs = mc_unique_pairs + fillin_label_pairs
print(f'  At batch size {batch_size}: ~{total_pairs // batch_size + 1} API calls')
print(f'  Estimated tokens (input ~200 + output ~50 per pair): ~{total_pairs * 250 / 1000:.0f}K tokens')


### 10.7 Sample Cleaned LLM Labeling Inputs

What the LLM will actually see for each (problem, wrong answer) pair — after HTML and MathML stripping.


In [ ]:
# ── MC example ───────────────────────────────────────────────────
mc_ex_id = mc_wrong_text.groupby('problem_id').size().sort_values(ascending=False).index[2]
mc_ex = problems[problems['problem_id'] == mc_ex_id].iloc[0]
mc_opts = [strip_html(o) for o in str(mc_ex['Multiple Choice Options']).split('||')]
mc_correct = strip_html(mc_ex['Multiple Choice Answers'])
mc_distractors = [o for o in mc_opts if normalise_answer(o) != normalise_answer(mc_correct)]

print('=== MC LABELING INPUT ===')
print(f'Problem : {strip_html(mc_ex["Problem Body"])[:300]}')
print(f'Correct : {mc_correct}')
print(f'Distractors to label:')
for d in mc_distractors:
    cnt = mc_wrong_text[mc_wrong_text['problem_id']==mc_ex_id][mc_wrong_text['answer_text'].str.strip()==d.strip()].shape[0]
    print(f'  - "{d}"  (chosen by {cnt:,} students)')

print()

# ── Fill-in example ───────────────────────────────────────────────
per_prob = fillin_wrong.groupby('problem_id').size().sort_values(ascending=False)
fi_ex_id = per_prob.index[2]
fi_ex = problems[problems['problem_id'] == fi_ex_id].iloc[0]
fi_top10 = fillin_wrong[fillin_wrong['problem_id'] == fi_ex_id]['answer_text'].value_counts().head(10)

print('=== FILL-IN LABELING INPUT ===')
print(f'Problem : {strip_html(fi_ex["Problem Body"])[:300]}')
print(f'Correct : {fi_ex["Fill-in Answers"]}')
print(f'Top-10 wrong answers to label:')
for ans, cnt in fi_top10.items():
    print(f'  - "{ans}"  ({cnt:,} students)')


### 10.8 Stage 1 Readiness Summary

| Decision | Value |
|---|---|
| Misconception row definition | `discrete_score=0` AND `answer_text ≠ correct answer` |
| Total genuine misconception rows | 605,773 (32.5% of all interactions) |
| MC problems — LLM-labelable | 766 / 795 (96.4%, text-only options) |
| MC problems — skipped | 29 / 795 (3.6%, image options) |
| Fill-in labeling strategy | Top 10 wrong answers per problem |
| Fill-in top-10 coverage | ~74% of wrong attempts (mean) |
| MC unique (problem, distractor) pairs | 1,704 |
| Fill-in top-10 (problem, answer) pairs | 18,816 |
| Total unique pairs to label in Stage 1 | 20,520 |
| Estimated API calls (batch size 50) | ~411 |
| Estimated input tokens (~250/pair) | ~5.1M |
| Estimated output tokens (~50/pair) | ~1M |
| Model | Claude Sonnet (minimum) |
| Estimated cost (Sonnet: $3/M in, $15/M out) | ~$30 one-time |
| MathML handling | Strip via `strip_html()` before LLM input |
